In [54]:
import numpy as np
import torch

### PER formula
$Cost(v_{ref},v_{hyp}) = \frac{\sum_iI(v_{ref}^{(i)} \ne v_{hyp}^{(i)}) M^{(i)}}{\sum_i M^{(i)}}$ 

In [71]:
from torch.nn.utils.rnn import pad_sequence

# List of tensors with varying lengths along dim 0
seqs = [torch.tensor([1, 2, 3]), torch.tensor([4, 5])]
padded_batch = pad_sequence(seqs, batch_first=True, padding_value=0)


In [118]:
v_ref = [[1, 0, -1],[1, 0, -1],[1, 0, -1],[1, 0, -1],[1, 0, -1],[1, 0, -1],[1, 0, -1]]
v_hyp = [[ 0, -1],[1, -1],[-1, 0],[0, -1],[-1, -1],[-1],[ 0]]

def normalize(pred: list, ref: list):
    pred = pred + [0 for _ in range(len(ref)- len(pred))]
    return pred

def get_ter(pred : list, ref : list):

    pred_padded = torch.zeros_like(ref)
    pred_padded = pred_padded.put(torch.arange(0, pred.shape[0]), pred)
    
    indicator = (pred_padded != ref)
    
    m = (torch.abs(ref) + torch.abs(pred_padded)) > 0
    num = (indicator * m).sum()
    den = m.sum()
    return num / den


In [119]:
v_hyp_padded = []
for r, h in zip(v_ref, v_hyp):
    v_hyp_padded.append(normalize(h, r))

In [126]:
v_ref_ts = torch.tensor(v_ref)
v_hyp_padded_ts = torch.tensor(v_hyp_padded)

indicator = v_hyp_padded_ts != v_ref_ts

m = (torch.abs(v_ref_ts) + torch.abs(v_hyp_padded_ts)) > 0

num = (indicator * m).sum(dim=-1)
den = m.sum(dim=-1)
(num / den).mean()

tensor(0.9524)

In [ ]:
import unicodedata

# for each letter we write out the presence of stuff
# depending on the input the tone vector will change

# Vector:

def map_tone(char):
    # print(char)
    decomposed = unicodedata.normalize('NFD', char)

    if '\u0301' in decomposed: return 1
    elif '\u0300' in decomposed: return -1
    else: return 0

def extract_tone(word):
    word_decomposed = map(map_tone, list(word))
    return list(word_decomposed)


test_chars = ['áòmẹ', "crazy"]
test_chars_decomposed = list(map(extract_tone, test_chars))

In [ ]:
s = test_chars[0][0]
unicodedata.normalize('NFD',s)

'á'

In [187]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers

tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

tokenizer.normalizer = normalizers.Sequence(
    [
        normalizers.NFD()
    ]
)

special_tokens = ['\u0301', '\u0300']
vocab = {
    "[UNK]" : 0,
    "\u0301" : 1,
    "\u0300" : 2
}
tokenizer.pre_tokenizer = pre_tokenizers.Split(pattern="", behavior="removed")
tokenizer.model = models.WordPiece(vocab, unk_token="[UNK]")

In [150]:
from datasets import load_dataset

data = load_dataset("nolimitsxl/wazobia-asr", "afrispeech_yor", split="train", streaming=True)
data = data.select_columns("text")

In [97]:
sample = data.take(100)

In [ ]:
normalizers.N

In [ ]:
inst = next(iter(sample))

TypeError: TextInputSequence must be str

In [178]:
tok_inst = tokenizer.encode(inst["text"])
len(tok_inst.ids), len(inst["text"])

(29, 23)

In [192]:
from transformers import PreTrainedTokenizerFast

wrapped_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]"
)

In [182]:
for batch in sample.iter(batch_size=32):
  batch_encoded = tokenizer.encode_batch(batch["text"])

In [195]:
wrapped_tokenizer.encode(batch["text"][0])

[0,
 2,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 2,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 2,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 2,
 0,
 0,
 2,
 0,
 0,
 1,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 1,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 2,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 2,
 0,
 0,
 0,
 0,
 0]

In [190]:
batch_encoded

[Encoding(num_tokens=222, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=37, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=133, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=283, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [98]:
def extract_tone_ds(row):
    word = row["text"]
    word_decomposed = list(map(map_tone, word))
    assert len(word) == len(word_decomposed)
    row["text_decomposed"] = torch.tensor(word_decomposed)
    return row


In [99]:
sample_processed = sample.map(extract_tone_ds)

In [62]:
for inst in sample_processed:
    
    print(f" Text {inst["text"]} \t Extracted Tone {inst["text_decomposed"]}", end="\n")

 Text máa jẹ oúnjẹ aṣaralóore 	 Extracted Tone tensor([0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0])
 Text ẹni tí inú rẹ̀ ń dùn máa ń jẹ àsè nígbà gbogbo òwe 	 Extracted Tone tensor([ 0,  0,  0,  0,  0,  1,  0,  0,  0,  1,  0,  0,  0, -1,  0,  1,  0,  0,
        -1,  0,  0,  0,  1,  0,  0,  1,  0,  0,  0,  0, -1,  0, -1,  0,  0,  1,
         0,  0, -1,  0,  0,  0,  0,  0,  0,  0,  0, -1,  0,  0])
 Text ní tímótì bíbélì sọ pé 'nǹkan máa le gan an ó sì máa nira' lákòókò tá a wà yìí torí náà kò yà mí lẹ́nu bí ọrọ̀ ajé ṣe ń dẹnu kọlẹ̀ tí nǹkan sì ń gbówó lórí kí nǹkan lè rọrùn fún mi mi ò kì í náwó sórí àwọn nǹkan tí kò ṣe pàtàkì pha ilya a erbaijan 	 Extracted Tone tensor([ 0,  1,  0,  0,  1,  0,  1,  0, -1,  0,  0,  1,  0,  1,  0, -1,  0,  0,
         0,  0,  0,  1,  0,  0,  0, -1,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0, -1,  0,  0,  1,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  1,  0, -1,  1,  0, -1,  0,  0,  

In [198]:
!pip install ekko_bench

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


ERROR: Could not find a version that satisfies the requirement ekko_bench (from versions: none)

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: C:\Python312\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for ekko_bench


ImportError: To be able to use ter, you need to install the following dependencies['ekko_bench'] using 'pip install ekko_bench' for instance'

In [4]:
a= [[1,2,3,5],[1,3,5,2,],[4,6,7,9,0]]
{x for row in a for x in row}

{0, 1, 2, 3, 4, 5, 6, 7, 9}